# DASAD — demonstration pipeline

### The notebook relies directly on the YAML configuration and the implementation found in the `DASAD/` directory.

- `M0 + S0` is trained exclusively on the first `SOURCE_SIZE` source samples;
- the drift detector always receives data scaled by `S0`;
- upon drift detection, target samples are first processed using the previous model–scaler pair;
- once a full raw target window is collected, the engine fits a new scaler `Sk`;
- the new `Mk + Sk` pair is activated only for the subsequent, unseen sample;
- target labels are used solely for final evaluation.


## 1. Imports, seed and logging


In [1]:
import logging
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import yaml
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    average_precision_score,
    roc_auc_score,
)
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from imblearn.metrics import geometric_mean_score

from DASAD.detectors import DriftDetector
from DASAD.engine import StreamEngine
from DASAD.models import DANNPredictor
from DASAD.networks import build_encoder, build_task, build_discriminator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception as error:
    print("Deterministic TensorFlow operations unavailable:", error)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,
)

pd.set_option("display.max_columns", 100)


2026-09-21 09:55:57.423473: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-21 09:55:57.455696: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-21 09:55:57.456312: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-21 09:55:57.998543: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT



## 2. Loading the YAML configuration

All parameters for the model, network, detector, and engine are sourced from the same file as in `main.py`.


In [2]:
CONFIG_PATH = Path("configs/config_ds1.yaml")

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

SOURCE_SIZE = int(config["engine"]["source_size"])
TARGET_WINDOW_SIZE = int(config["engine"]["window_size"])

hyperparameters = {
    "learning_rate": config["predictor"]["learning_rate"],
    "encoder_learning_rate": config["network"]["encoder"]["learning_rate"],
    "discriminator_learning_rate": config["network"]["discriminator"]["discriminator_learning_rate"],
    "latent_dim": config["network"]["encoder"]["latent_dim"],
    "gamma": config["predictor"]["gamma"],
    "alpha": config["predictor"]["alpha"],
    "lambda": config["predictor"].get("lambda", config["predictor"].get("adv_weight")),
    "batch_size": config["predictor"]["batch_size"],
    "epochs": config["predictor"]["epochs"],
    "source_size": SOURCE_SIZE,
    "target_window_size": TARGET_WINDOW_SIZE,
}

print("Configuration:", CONFIG_PATH)
display(pd.Series(hyperparameters, name="value").to_frame())


Configuration: configs/config_ds1.yaml


,value
learning_rate,0.00030
encoder_learning_rate,0.00010
discriminator_learning_rate,0.00001
latent_dim,8.00000
gamma,2.00000
alpha,0.50000
lambda,0.00003
batch_size,128.00000
epochs,120.00000
source_size,3000.00000




## 3. Loading raw data.


In [3]:
DATA_PATH = Path(config["data"]["path"])
df = pd.read_csv(DATA_PATH)

x_cols = config["x_cols"]
y_col = config["y_col"]

X_raw = df[x_cols].copy()
y = df[y_col].copy()

if not 0 < SOURCE_SIZE < len(X_raw):
    raise ValueError("source_size must be between 1 and len(dataset) - 1")

print("Dataset path:", DATA_PATH)
print("Full dataset shape:", df.shape)
print("Raw feature shape:", X_raw.shape)
print("Labels shape:", y.shape)
print("Source range: 0 ..", SOURCE_SIZE - 1)
print("Source samples:", SOURCE_SIZE)
print("Future stream samples:", len(X_raw) - SOURCE_SIZE)
print("Class counts in source:")
display(y.iloc[:SOURCE_SIZE].value_counts().sort_index().rename("count").to_frame())


Dataset path: data/tcm_3_anomaly4_v1.csv
Full dataset shape: (9367, 57)
Raw feature shape: (9367, 29)
Labels shape: (9367,)
Source range: 0 .. 2999
Source samples: 3000
Future stream samples: 6367
Class counts in source:


,count
class,
0,2878
1,122


## 4. Matching the initial scaler S0

`S0` sees only the source. Subsequent scalers will be matched within the engine only after the corresponding target windows have been collected.


In [4]:
source_scaler = StandardScaler()
source_scaler.fit(X_raw.iloc[:SOURCE_SIZE].to_numpy())

X_source_scaled = source_scaler.transform(X_raw.iloc[:SOURCE_SIZE])

print("Scaler S0:", type(source_scaler).__name__)
print("Fitted samples:", SOURCE_SIZE)
print("Scaled source shape:", X_source_scaled.shape)
print("Scaled source min/max:", X_source_scaled.min(), X_source_scaled.max())
print("Constant source features:", int(np.sum(np.ptp(X_raw.iloc[:SOURCE_SIZE].to_numpy(), axis=0) == 0)))


Scaler S0: StandardScaler
Fitted samples: 3000
Scaled source shape: (3000, 29)
Scaled source min/max: -7.779676613916222 6.675878068737523
Constant source features: 0


/home/jovyan/.conda/envs/domain_adapt_stream/lib/python3.11/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


## 5. Building the network from a YAML configuration

The encoder concludes with a linear latent representation and `LayerNormalization`. The latent dimension must be identical for the encoder, the task classifier, and the discriminator.


In [5]:
latent_dims = {
    int(config["network"][name]["latent_dim"])
    for name in ("encoder", "task", "discriminator")
}
assert len(latent_dims) == 1, "All network components must use the same latent_dim"

input_shape = (len(x_cols),)
encoder = build_encoder(input_shape, config["network"]["encoder"])
task = build_task(config["network"]["task"])
discriminator = build_discriminator(config["network"]["discriminator"])

print("ENCODER")
encoder.summary()
print()
print("TASK CLASSIFIER")
task.summary()
print()
print("DOMAIN DISCRIMINATOR")
discriminator.summary()


2026-09-21 09:55:59.265579: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


ENCODER
Model: "encoder"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                1920      
                                                                 
 layer_normalization (Layer  (None, 64)                128       
 Normalization)                                                  
                                                                 
 leaky_re_lu (LeakyReLU)     (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 layer_normalization_1 (Lay  (None, 32)                64        
 erNormalization)                                                
                                                                 
 leaky_re_lu_1 (LeakyReLU)   (None, 32)            

## 6. Drift Detector and DANN

The `lambda` value from the YAML file is passed as `adv_weight` to the `DANNPredictor`. The legacy Adam optimizer is used within the implementation so that ADAPT can update different parts of the DANN without encountering variable recognition errors.


In [6]:
detector_config = config["detector"]
predictor_config = config["predictor"]

detector = DriftDetector(
    reference_size=detector_config["reference_size"],
    window_size=detector_config["window_size"],
    min_instances=detector_config["min_instances"],
    delta=detector_config["delta"],
    threshold=detector_config["threshold"],
    alpha=detector_config["alpha"],
)

predictor = DANNPredictor(
    encoder=encoder,
    task=task,
    discriminator=discriminator,
    adv_weight=predictor_config.get("lambda", predictor_config.get("adv_weight", 0.001)),
    learning_rate=predictor_config["learning_rate"],
    gamma=predictor_config["gamma"],
    class_balance=predictor_config["class_balance"],
    alpha=predictor_config["alpha"],
    epochs=predictor_config["epochs"],
    batch_size=predictor_config["batch_size"],
    threshold=predictor_config["threshold"],
    pretrain_epochs=predictor_config.get("pretrain_epochs", predictor_config["epochs"]),
    pretrain_validation_split=predictor_config.get("pretrain_validation_split", 0.2),
    pretrain_patience=predictor_config.get("pretrain_patience", 20),
)

print("DANN lambda:", predictor.adv_weight)
print("DANN epochs:", predictor.epochs)
print("DANN batch size:", predictor.batch_size)


DANN lambda: 3e-05
DANN epochs: 120
DANN batch size: 128


## 7. Stream Startup

We feed the raw `X_raw` into the engine. During initialization, M0 is trained on the source data scaled by S0. Following each alarm, the engine collects the raw target data, fits a new scaler, and activates a new model–scaler pair.


In [7]:
engine = StreamEngine(
    detector=detector,
    predictor=predictor,
    source_data=X_raw.iloc[:SOURCE_SIZE].to_numpy(),
    source_labels=y.iloc[:SOURCE_SIZE].to_numpy(),
    source_size=SOURCE_SIZE,
    window_size=TARGET_WINDOW_SIZE,
    source_scaler=source_scaler,
    feature_names=x_cols,
)

results = engine.run(X_raw, y)

print()
print("Run completed")
print("Predictions:", len(results["predictions"]))
print("Detected drifts:", results["drift_points"])
print("Number of adaptations:", len(results["model_trainings"]) - 1)
print("Number of fitted scalers:", len(results["scaler_updates"]))


2026-09-21 09:55:59,533 [INFO] DASAD.engine: ENGINE INITIALIZED | source_raw=(3000, 29) | source_scaled=(3000, 29) | source_size=3000 | target_window_size=50 | features=29
2026-09-21 09:55:59,533 [INFO] DASAD.engine: SCALER FITTED | name=S0 source scaler | class=StandardScaler | samples=3000 | features=29 | raw_min=0.0561083 | raw_max=8.51027e+06 | constant_features=0 | mean_feature_range=232120
2026-09-21 09:55:59,574 [INFO] DASAD.models: ======================================================================
2026-09-21 09:55:59,574 [INFO] DASAD.models: MODEL SUMMARY: M0 - initial source anomaly detector
2026-09-21 09:55:59,575 [INFO] DASAD.models: ======================================================================
2026-09-21 09:55:59,575 [INFO] DASAD.models: Model: "model"
2026-09-21 09:55:59,575 [INFO] DASAD.models: _________________________________________________________________
2026-09-21 09:55:59,575 [INFO] DASAD.models:  Layer (type)                Output Shape              P


Run completed
Predictions: 6367
Detected drifts: [7579, 8527]
Number of adaptations: 2
Number of fitted scalers: 3


## 8. Summary of models and scalers

`model_trainings` records dataset sizes, window indices, and the relative change in weights. `scaler_updates` records the exact sample range used to fit each scaler.

In [8]:
model_training_df = pd.DataFrame(results["model_trainings"])

model_columns = [
    column for column in [
        "model", "previous_model", "training_type", "adaptation_round",
        "source_samples", "target_samples", "drift_index", "target_start",
        "target_end", "source_scaler", "target_scaler", "epochs_requested",
        "epochs_completed", "elapsed_seconds", "encoder_weight_relative_delta",
        "task_weight_relative_delta", "discriminator_weight_relative_delta",
    ]
    if column in model_training_df.columns
]

display(model_training_df[model_columns])

scaler_summary_df = pd.DataFrame(results["scaler_updates"])[
    ["scaler", "scaler_class", "fitted_samples", "fitted_start", "fitted_end"]
]
display(scaler_summary_df)


,model,previous_model,training_type,adaptation_round,source_samples,target_samples,drift_index,target_start,target_end,source_scaler,target_scaler,epochs_requested,epochs_completed,elapsed_seconds,encoder_weight_relative_delta,task_weight_relative_delta,discriminator_weight_relative_delta
0,M0,NaN,source_pretraining,NaN,3000,0,NaN,NaN,NaN,NaN,NaN,NaN,40.0,3.001304,NaN,NaN,NaN
1,M1,M0,dann_adaptation,1.0,3000,50,7579.0,7579.0,7628.0,S0,S1,120.0,NaN,9.899521,0.119460,0.251783,1.060885
2,M2,M1,dann_adaptation,2.0,3000,50,8527.0,8527.0,8576.0,S0,S2,120.0,NaN,9.129651,0.131113,0.196334,0.476036


,scaler,scaler_class,fitted_samples,fitted_start,fitted_end
0,S0,StandardScaler,3000,0,2999
1,S1,StandardScaler,50,7579,7628
2,S2,StandardScaler,50,8527,8576


## 9. Prequential metrics

These are official online results. Samples from the adaptive window retain the predictions of the previous model—they are not retrospectively replaced.

In [9]:
y_true = np.asarray(results["true"]).reshape(-1)
y_pred = np.asarray(results["predictions"]).reshape(-1)

print("F1:", f1_score(y_true, y_pred, zero_division=0))
print("G-mean:", geometric_mean_score(y_true, y_pred))
print("Predicted anomalies:", int(y_pred.sum()))
print("True anomalies:", int(y_true.sum()))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


F1: 0.8596881959910913
G-mean: 0.8765247776294388
Predicted anomalies: 198
True anomalies: 251
Confusion matrix:
[[6111    5]
 [  58  193]]


## 10. Results of successive model versions

The model changes only starting from the sample following the end of the target window. The table below assigns the model and scaler versions available online to each official prediction.


In [10]:
prediction_df = pd.DataFrame({
    "idx": results["idx"],
    "true": results["true"],
    "prediction": results["predictions"],
})

prediction_df["model"] = "M0"
prediction_df["scaler"] = "S0"

adaptations = [
    item for item in results["model_trainings"]
    if item.get("training_type") == "dann_adaptation"
]

for item in adaptations:
    activation_index = int(item["target_end"]) + 1
    mask = prediction_df["idx"] >= activation_index
    prediction_df.loc[mask, "model"] = item["model"]
    prediction_df.loc[mask, "scaler"] = item["target_scaler"]

per_model_results = []
for (model_name, scaler_name), group in prediction_df.groupby(["model", "scaler"], sort=False):
    row = {
        "model": model_name,
        "scaler": scaler_name,
        "start": int(group["idx"].min()),
        "end": int(group["idx"].max()),
        "samples": len(group),
        "true_anomalies": int(group["true"].sum()),
        "predicted_anomalies": int(group["prediction"].sum()),
        "f1": f1_score(group["true"], group["prediction"], zero_division=0),
        "g_mean": geometric_mean_score(group["true"], group["prediction"]),
    }
    per_model_results.append(row)

per_model_df = pd.DataFrame(per_model_results)
display(per_model_df)


,model,scaler,start,end,samples,true_anomalies,predicted_anomalies,f1,g_mean
0,M0,S0,3000,7628,4629,185,161,0.907514,0.920807
1,M1,S1,7629,8576,948,36,16,0.576923,0.645143
2,M2,S2,8577,9366,790,30,21,0.823529,0.836660
